# Session 7 — LangGraph for Real Users: Streaming, Human-in-the-Loop, Time Travel, Memory & Subgraphs

In Session 6 you learned what a graph *is* — **state, nodes, edges** — and how to make it **remember** with a checkpointer and a store. That makes a graph *work*.

This session makes a graph **usable by real people and safe to run in production**. Five capabilities, each solving a problem you hit the moment a graph leaves your laptop:

| Problem | Capability | Doc |
|---|---|---|
| The screen is blank for 8 seconds — did it crash? | **Streaming** | `streaming` |
| I need every token, tool call, and step for a live UI | **Event streaming** | `event-streaming` |
| A human must approve before money moves / a grade changes | **Interrupts** | `interrupts` |
| I want to debug a bad run, or explore "what if we'd chosen X?" | **Time travel** | `use-time-travel` |
| Remember this student across sessions without blowing the context | **Memory** | `add-memory` |
| My graph is huge — I want reusable, testable pieces | **Subgraphs** | `use-subgraphs` |

**How we'll learn it:** with examples from a world you already know — a **university**: enrolling in a course, an academic advisor, a grade-change approval, essay feedback, an admissions pipeline. Each one has a direct parallel in our **Northstar** support project, which we'll call out as we go.

Everything below is verified on **LangGraph 1.2.7** with the same cheap Gemini model from Sessions 3–6.


## 1. Setup

🎯 **Purpose:** install LangGraph and confirm the Gemini key is available — same model and `.env` as every prior session.

> If an import still uses an older version after the first install, restart the kernel and run the imports again.


In [1]:
%pip install -q -U langgraph langchain langchain-google-genai python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)  # reads .env in this folder; never hard-code keys

import os

MODEL = "google_genai:gemini-3.1-flash-lite"  # one place to swap the model

print("GEMINI_API_KEY:", "set" if os.getenv("GEMINI_API_KEY") else "MISSING")
print("Model:", MODEL)

GEMINI_API_KEY: set
Model: google_genai:gemini-3.1-flash-lite


In [3]:
# Imports we reuse across the whole notebook.
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL)
print("Ready.")

Ready.


## 2. Streaming — show progress as the graph runs

A graph can take several seconds: LLM calls, tool calls, multiple nodes. If the user stares at a blank screen the whole time, the app feels broken. **Streaming** emits results *as each step happens* instead of only at the end.

💡 **Real-life:** a **pizza tracker** — "Preparing → Baking → Out for delivery" — instead of a blank page until the doorbell rings. Same total wait, completely different experience.

You choose *what* to stream with `stream_mode`. We'll try each one on a tiny **course-enrollment** graph.

Reference: [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming).


### 2.1 A graph to stream

🎯 **Purpose:** three quick steps — check prerequisites, check seats, confirm. Nothing fancy; we just want something with visible stages to watch.


In [4]:
class EnrollState(TypedDict):
    student: str
    course: str
    prereqs_ok: bool
    seats_ok: bool
    status: str


def check_prereqs(state: EnrollState) -> dict:
    return {"prereqs_ok": True}


def check_seats(state: EnrollState) -> dict:
    return {"seats_ok": True}


def confirm(state: EnrollState) -> dict:
    ok = state["prereqs_ok"] and state["seats_ok"]
    return {"status": "enrolled" if ok else "waitlisted"}


enroll = (
    StateGraph(EnrollState)
    .add_node("check_prereqs", check_prereqs)
    .add_node("check_seats", check_seats)
    .add_node("confirm", confirm)
    .add_edge(START, "check_prereqs")
    .add_edge("check_prereqs", "check_seats")
    .add_edge("check_seats", "confirm")
    .add_edge("confirm", END)
    .compile()
)

request = {"student": "Aisha", "course": "CS101"}
print("Graph ready.")

Graph ready.


### 2.2 `stream_mode="updates"` — see each step finish

🎯 **Purpose:** the most useful mode for progress. After each node runs, you get `{node_name: the fields it changed}`. Perfect for "Checked prerequisites ✓ … Checked seats ✓ … Enrolled ✓".


In [5]:
for chunk in enroll.stream(request, stream_mode="updates"):
    for node_name, changed in chunk.items():
        print(f"{node_name:15} -> {changed}")

check_prereqs   -> {'prereqs_ok': True}
check_seats     -> {'seats_ok': True}
confirm         -> {'status': 'enrolled'}


### 2.3 `stream_mode="values"` — the full state after each step

🎯 **Purpose:** instead of just the change, get the **entire state** after every step. Great for debugging — you watch the shared notebook fill in field by field.


In [6]:
for snapshot in enroll.stream(request, stream_mode="values"):
    print(snapshot)

{'student': 'Aisha', 'course': 'CS101'}
{'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True}
{'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True, 'seats_ok': True}
{'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True, 'seats_ok': True, 'status': 'enrolled'}


### 2.4 Combine modes — pass a list

🎯 **Purpose:** ask for more than one mode at once. Now each item is a **`(mode, chunk)` tuple** so you can tell them apart.


In [7]:
for mode, chunk in enroll.stream(request, stream_mode=["updates", "values"]):
    print(f"[{mode}] {chunk}")

[values] {'student': 'Aisha', 'course': 'CS101'}
[updates] {'check_prereqs': {'prereqs_ok': True}}
[values] {'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True}
[updates] {'check_seats': {'seats_ok': True}}
[values] {'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True, 'seats_ok': True}
[updates] {'confirm': {'status': 'enrolled'}}
[values] {'student': 'Aisha', 'course': 'CS101', 'prereqs_ok': True, 'seats_ok': True, 'status': 'enrolled'}


### 2.5 `stream_mode="messages"` — type out the answer token by token

🎯 **Purpose:** the "ChatGPT effect". When a node calls an LLM, this mode streams the reply **one chunk at a time** as `(token, metadata)`. `token.text` is the piece of text; `metadata["langgraph_node"]` tells you which node produced it.

💡 **Real-life:** a tutor writing the answer on the whiteboard while you watch, instead of handing you a finished page.


In [8]:
def tutor(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


tutor_graph = (
    StateGraph(MessagesState)
    .add_node("tutor", tutor)
    .add_edge(START, "tutor")
    .add_edge("tutor", END)
    .compile()
)

question = {"messages": [{"role": "user", "content": "In one sentence, what is a prime number?"}]}
for token, metadata in tutor_graph.stream(question, stream_mode="messages"):
    print(token.text, end="", flush=True)   # arrives in pieces, not all at once
print()

A prime number is a whole number greater than 1 that has no positive divisors other than 1 and itself.


### 2.6 `stream_mode="custom"` — emit your own progress messages

🎯 **Purpose:** sometimes the useful update isn't a token or a state change — it's *your* status line: "Searching the catalog…", "Found 3 courses". Call `get_stream_writer()` inside a node and `write()` whatever you want; it comes out on the `"custom"` stream.

💡 **Real-life:** a long download that prints "Connecting… / Downloading… / Verifying…" so you know it's alive.


In [9]:
from langgraph.config import get_stream_writer


class SearchState(TypedDict):
    query: str
    results: list


def search_catalog(state: SearchState) -> dict:
    writer = get_stream_writer()               # a hook to the "custom" stream
    writer({"progress": "Searching the course catalog..."})
    matches = ["CS101", "CS102", "CS201"]
    writer({"progress": f"Found {len(matches)} matching courses"})
    return {"results": matches}


search_graph = (
    StateGraph(SearchState)
    .add_node("search_catalog", search_catalog)
    .add_edge(START, "search_catalog")
    .add_edge("search_catalog", END)
    .compile()
)

for chunk in search_graph.stream({"query": "intro CS"}, stream_mode="custom"):
    print(chunk["progress"])

Searching the course catalog...
Found 3 matching courses


### 2.7 Which stream mode?

| `stream_mode` | You get | Reach for it to… |
|---|---|---|
| `"updates"` | `{node: changed fields}` after each node | Show step-by-step progress (most common) |
| `"values"` | The full state after each step | Debug / watch state grow |
| `"messages"` | `(token, metadata)` | Type the LLM answer out live |
| `"custom"` | Whatever you `write()` | Emit your own status lines |
| `["updates", ...]` | `(mode, chunk)` tuples | Combine several of the above |

**Northstar parallel:** while the support agent looks up an invoice and searches policy, stream `"custom"` progress ("Checking your account… Reading the refund policy…") and `"messages"` tokens so the customer sees a live, trustworthy reply instead of a spinner.


## 3. Event streaming — the most detailed view (`astream_events`)

`stream()` gives you a handful of tidy modes. Sometimes you need **everything**: the model started, a token arrived, a tool was called, the tool returned, a node finished. `astream_events` is a single fine-grained stream of **every** event inside the graph — ideal for building a rich UI that shows tool calls *and* tokens *and* steps together.

💡 **Real-life:** live match commentary ("kick-off… pass… shot… goal!") versus only being told the final score.

Two things to know:

- It is **async** — use `async for`. In a notebook you can `await` at the top level of a cell (no `asyncio.run`).
- Each event is a dict: `event["event"]` (the type), `event["name"]` (which model/tool/node), and `event["data"]` (the payload).

Reference: [Event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming).


### 3.1 Watch an agent think, call a tool, and answer

🎯 **Purpose:** give an **academic advisor** agent one tool (`lookup_course`) and stream its events. We'll pick out three kinds: a tool starting, a tool finishing, and the answer tokens.


In [10]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def lookup_course(code: str) -> str:
    """Look up a university course by its code, e.g. CS101."""
    catalog = {
        "CS101": "Intro to Computer Science — Mondays 9am — 3 credits",
        "MATH200": "Linear Algebra — Tuesdays 11am — 4 credits",
    }
    return catalog.get(code.upper(), f"No course found for {code}")


advisor = create_agent(
    model=MODEL,
    tools=[lookup_course],
    system_prompt="You are a university academic advisor. Use tools for course facts.",
)
print("Advisor ready.")

Advisor ready.


In [11]:
# astream_events is async -> use `async for`, and `await` the coroutine at top level.
async def watch_advisor(user_text: str):
    counts = {}
    async for event in advisor.astream_events(
        {"messages": [{"role": "user", "content": user_text}]},
        version="v2",                       # the stable event format in LangGraph 1.x
    ):
        kind = event["event"]
        counts[kind] = counts.get(kind, 0) + 1

        if kind == "on_tool_start":
            print(f"🔧 calling tool  {event['name']}({event['data'].get('input')})")
        elif kind == "on_tool_end":
            result = event["data"].get("output")
            text = getattr(result, "content", result)   # a ToolMessage -> its text
            print(f"✅ tool returned {text}")
        elif kind == "on_chat_model_stream":
            piece = event["data"]["chunk"].text
            if piece:
                print(piece, end="", flush=True)

    print("\n\nevent types seen:", {k: counts[k] for k in sorted(counts)})


await watch_advisor("When does CS101 meet?")

🔧 calling tool  lookup_course({'code': 'CS101'})
✅ tool returned Intro to Computer Science — Mondays 9am — 3 credits
CS101 (Intro to Computer Science) meets on Mondays at 9am.

event types seen: {'on_chain_end': 4, 'on_chain_start': 4, 'on_chain_stream': 6, 'on_chat_model_end': 2, 'on_chat_model_start': 2, 'on_chat_model_stream': 7, 'on_tool_end': 1, 'on_tool_start': 1}


### 3.2 The events you'll use most

| Event | Fires when… |
|---|---|
| `on_chain_start` / `on_chain_end` | a node (or the whole graph) starts / finishes |
| `on_chat_model_start` | the LLM is invoked |
| `on_chat_model_stream` | one token of the LLM reply arrives |
| `on_chat_model_end` | the LLM finishes |
| `on_tool_start` / `on_tool_end` | a tool is called / returns |

**`stream()` vs `astream_events()` — which one?**

- Start with **`stream()`**: it's synchronous and its modes cover most needs (progress, tokens, custom lines).
- Reach for **`astream_events()`** when you're building a **detailed UI** that must show tool calls, tokens, and step boundaries *together* — or when you need to react to a specific low-level event.

**Northstar parallel:** the support UI shows "🔧 Looking up invoice INV-5013… ✅ Found it…" while the final answer types out — every `on_tool_start`/`on_tool_end`/`on_chat_model_stream` becomes a line the customer can see.


## 4. Interrupts — pause for a human (human-in-the-loop)

Some steps must **not** happen automatically: moving money, changing a grade, deleting an account. LangGraph lets a node **pause the whole graph**, hand a question to a human, and **resume** with their answer.

💡 **Real-life:** a **grade change**. A teaching assistant proposes bumping a B+ to an A−, but the *professor* must sign off before it's recorded. The process stops, waits — maybe overnight — then continues with the decision.

Two ingredients (both from Session 6's persistence toolkit):

- a **checkpointer** — so the paused state is saved while we wait;
- a **`thread_id`** — so we resume the *right* case.

Reference: [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts).


### 4.1 Anatomy of an interrupt

1. Inside a node, call **`interrupt(payload)`** — `payload` is any data the human needs to decide. The graph **pauses** right there.
2. `invoke(...)` returns a dict with an **`__interrupt__`** key holding what you passed.
3. A human decides. You resume by calling the graph again with **`Command(resume=<their answer>)`** on the **same `thread_id`**.
4. `interrupt(...)` then **returns** that answer, and the node continues.

⚠️ **The one gotcha:** on resume, the paused node **re-runs from its first line** — everything *before* `interrupt()` executes again. So put `interrupt()` early, and don't do irreversible work before it.


In [13]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


class GradeState(TypedDict):
    student: str
    old_grade: str
    new_grade: str
    decision: str


def professor_approval(state: GradeState) -> dict:
    # Pause and hand the professor everything they need to decide.
    approved = interrupt({
        "question": "Approve this grade change?",
        "student": state["student"],
        "change": f"{state['old_grade']} -> {state['new_grade']}",
    })
    # This line runs only AFTER a human resumes with Command(resume=...).
    return {"decision": "applied" if approved else "rejected"}


grade_graph = (
    StateGraph(GradeState)
    .add_node("professor_approval", professor_approval)
    .add_edge(START, "professor_approval")
    .add_edge("professor_approval", END)
    .compile(checkpointer=InMemorySaver())    # interrupts REQUIRE a checkpointer
)
print("Grade graph ready.")

Grade graph ready.


In [14]:
config = {"configurable": {"thread_id": "grade-req-1"}}

# 1) Start the graph — it runs until interrupt(), then PAUSES and returns.
paused = grade_graph.invoke(
    {"student": "Aisha", "old_grade": "B+", "new_grade": "A-"},
    config,
)
print("Paused. The professor is asked:")
print("  ", paused["__interrupt__"][0].value)
print("Graph is waiting — nothing has been applied yet.")

Paused. The professor is asked:
   {'question': 'Approve this grade change?', 'student': 'Aisha', 'change': 'B+ -> A-'}
Graph is waiting — nothing has been applied yet.


In [15]:
# 2) The professor approves. Resume the SAME thread with their answer.
resumed = grade_graph.invoke(Command(resume=True), config)
print("After approval  ->", resumed["decision"])

# A different request that gets rejected (a fresh thread_id = a fresh case).
reject_cfg = {"configurable": {"thread_id": "grade-req-2"}}
grade_graph.invoke({"student": "Ben", "old_grade": "C", "new_grade": "A"}, reject_cfg)
rejected = grade_graph.invoke(Command(resume=False), reject_cfg)
print("After rejection ->", rejected["decision"])

After approval  -> applied
After rejection -> rejected


**Northstar parallel:** a refund above a threshold pauses with `interrupt({"invoice": ..., "amount": ...})`. A human agent approves or denies, and `Command(resume=decision)` either processes the refund or sends a polite decline — the model never moves money on its own.


## 5. Time travel — inspect, replay, and fork the past

Because a checkpointer saves state after **every** step (Session 6), the whole history of a run is sitting there. **Time travel** uses it three ways: **list** the checkpoints, **replay** from any one of them, or **fork** — change the state at a past point and run a new branch to explore "what if?".

💡 **Real-life:** a **save-and-rewind** in a video game. Reload an earlier save, make a *different* choice, and play out an alternate ending — your original playthrough is untouched.

We'll use a two-step **essay-feedback** graph: pick a topic, then write feedback.

Reference: [Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel).


In [16]:
class EssayState(TypedDict):
    topic: str
    feedback: str


def pick_topic(state: EssayState) -> dict:
    return {"topic": state.get("topic") or "climate policy"}


def write_feedback(state: EssayState) -> dict:
    return {"feedback": f"Great argument about {state['topic']}. Add one counter-example."}


essay_graph = (
    StateGraph(EssayState)
    .add_node("pick_topic", pick_topic)
    .add_node("write_feedback", write_feedback)
    .add_edge(START, "pick_topic")
    .add_edge("pick_topic", "write_feedback")
    .add_edge("write_feedback", END)
    .compile(checkpointer=InMemorySaver())
)

cfg = {"configurable": {"thread_id": "essay-1"}}
first = essay_graph.invoke({"topic": "renewable energy"}, cfg)
print("Original feedback:", first["feedback"])

Original feedback: Great argument about renewable energy. Add one counter-example.


### 5.1 List the history

🎯 **Purpose:** `get_state_history(config)` returns every checkpoint, **newest first**. Each snapshot has `.values` (the state then), `.next` (which node would run next), and `.config` (its address, including a `checkpoint_id`).


In [17]:
history = list(essay_graph.get_state_history(cfg))
for snap in history:
    print(f"next={str(snap.next):22}  topic={snap.values.get('topic')}")

next=()                      topic=renewable energy
next=('write_feedback',)     topic=renewable energy
next=('pick_topic',)         topic=renewable energy
next=('__start__',)          topic=None


### 5.2 Replay from a past checkpoint

🎯 **Purpose:** pick the checkpoint taken **before** `write_feedback` ran, and resume from it with `invoke(None, checkpoint.config)`. Passing `None` as input means "don't add anything new — just continue from here."

⚠️ Replay **re-executes** the later nodes (it doesn't read a cache). With an LLM node, the model is called again and may answer slightly differently.


In [18]:
before_feedback = next(s for s in history if s.next == ("write_feedback",))
print("Rewinding to just before write_feedback (topic =", before_feedback.values["topic"], ")")

replayed = essay_graph.invoke(None, before_feedback.config)
print("Replayed feedback:", replayed["feedback"])

Rewinding to just before write_feedback (topic = renewable energy )
Replayed feedback: Great argument about renewable energy. Add one counter-example.


### 5.3 Fork — change the past and branch

🎯 **Purpose:** the powerful move. `update_state(checkpoint.config, {...})` writes a **new** checkpoint that edits the state at that point and returns a new config. Continue from it with `invoke(None, fork_config)` to get an **alternate** result — the original history stays intact.

💡 **Real-life:** "what if the student had written about *urban gardening* instead?" — rewind, swap the topic, and see the new feedback, without losing the first version.


In [19]:
fork_config = essay_graph.update_state(before_feedback.config, {"topic": "urban gardening"})
forked = essay_graph.invoke(None, fork_config)
print("Forked feedback:", forked["feedback"])

Forked feedback: Great argument about urban gardening. Add one counter-example.


**Northstar parallel:** a support answer went wrong. Rewind to the checkpoint before the reply node, `update_state` to correct the retrieved policy, and replay — you've reproduced and fixed the decision without re-running the whole conversation. Time travel is your debugger *and* an "edit-and-rerun" button for users.


## 6. Memory — remembering the right things (and forgetting the rest)

Session 6 introduced the two memories; here we use them well:

- **Short-term** = one conversation, saved by a **checkpointer** under a `thread_id`. Problem: conversations grow, and you can't feed 500 messages to the model forever. So we **trim**.
- **Long-term** = facts that outlive any single conversation, saved in a **store** under a namespace. We write a fact in one session and recall it in a **new** one.

💡 **Real-life:** short-term is *what we said in today's advising meeting*; long-term is *the file the advisor keeps about you between meetings* (your major, your year).

Reference: [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory).


### 6.1 Short-term: recall within a thread

🎯 **Purpose:** a quick confirmation of Session 6 — same `thread_id` means the graph remembers earlier turns.


In [20]:
chat = (
    StateGraph(MessagesState)
    .add_node("chat", lambda s: {"messages": [model.invoke(s["messages"])]})
    .add_edge(START, "chat")
    .add_edge("chat", END)
    .compile(checkpointer=InMemorySaver())
)

advisee = {"configurable": {"thread_id": "advisee-aisha"}}
chat.invoke({"messages": [{"role": "user", "content": "My major is Computer Science."}]}, advisee)
reply = chat.invoke({"messages": [{"role": "user", "content": "What is my major?"}]}, advisee)
print(reply["messages"][-1].text)

Your major is **Computer Science**.


### 6.2 Short-term: trim a long history

🎯 **Purpose:** if a conversation runs for hours, sending *every* message is slow and expensive — and can overflow the model's context window. `trim_messages` keeps only the most recent messages that fit a budget. You'd call it **inside a node**, right before the LLM, so the model always sees a bounded, recent slice.

💡 **Real-life:** you don't re-read the entire semester's emails before replying to one — you skim the last few.


In [21]:
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langchain_core.messages import HumanMessage, AIMessage

# Pretend this conversation has gone on for a while.
long_history = []
for i in range(8):
    long_history.append(HumanMessage(f"Question {i} about my class schedule."))
    long_history.append(AIMessage(f"Here is answer {i}."))

trimmed = trim_messages(
    long_history,
    strategy="last",                       # keep the most recent
    token_counter=count_tokens_approximately,
    max_tokens=60,                         # the budget
    start_on="human",                      # keep a valid conversation shape
)

print(f"Full history : {len(long_history)} messages")
print(f"After trim   : {len(trimmed)} messages (only these go to the model)")
for m in trimmed:
    print("   ", type(m).__name__, "->", m.content)

Full history : 16 messages
After trim   : 4 messages (only these go to the model)
    HumanMessage -> Question 6 about my class schedule.
    AIMessage -> Here is answer 6.
    HumanMessage -> Question 7 about my class schedule.
    AIMessage -> Here is answer 7.


### 6.3 Long-term: remember across sessions with a store

🎯 **Purpose:** write a durable fact in one thread and read it back in a **different** thread. The node reaches the store via `runtime.store` (injected by LangGraph), and `runtime.context` carries *whose* data this is — exactly the Session 6 pattern.

💡 **Real-life:** the advisor opens your file at the start of *every* meeting — a new conversation, the same remembered facts.


In [22]:
from dataclasses import dataclass
from langgraph.store.memory import InMemoryStore
from langgraph.runtime import Runtime


@dataclass
class Context:
    student_id: str


class ProfileState(TypedDict):
    note: str
    recalled: list


def remember(state: ProfileState, runtime: Runtime[Context]) -> dict:
    ns = ("university", "student", runtime.context.student_id, "profile")
    runtime.store.put(ns, "major", {"value": state["note"]})
    return {}


def recall(state: ProfileState, runtime: Runtime[Context]) -> dict:
    ns = ("university", "student", runtime.context.student_id, "profile")
    items = runtime.store.search(ns)
    return {"recalled": [item.value for item in items]}


store = InMemoryStore()                     # one store, shared by both graphs

writer_graph = (
    StateGraph(ProfileState, context_schema=Context)
    .add_node("remember", remember)
    .add_edge(START, "remember")
    .add_edge("remember", END)
    .compile(checkpointer=InMemorySaver(), store=store)
)
reader_graph = (
    StateGraph(ProfileState, context_schema=Context)
    .add_node("recall", recall)
    .add_edge(START, "recall")
    .add_edge("recall", END)
    .compile(checkpointer=InMemorySaver(), store=store)
)

In [23]:
# Monday's session writes the fact...
writer_graph.invoke(
    {"note": "Computer Science, class of 2027"},
    {"configurable": {"thread_id": "session-monday"}},
    context=Context(student_id="S-42"),
)

# ...Friday's session is a DIFFERENT thread, yet still recalls it.
out = reader_graph.invoke(
    {"note": ""},
    {"configurable": {"thread_id": "session-friday"}},
    context=Context(student_id="S-42"),
)
print("Recalled in a new session:", out["recalled"])

Recalled in a new session: [{'value': 'Computer Science, class of 2027'}]


| | Short-term (checkpointer) | Long-term (store) |
|---|---|---|
| Scope | One conversation (`thread_id`) | Across conversations (namespace) |
| Holds | The running message history | Durable facts you choose to keep |
| Growth problem | Trim / summarize old messages | Store only what matters |

**Northstar parallel:** short-term keeps this ticket's back-and-forth; long-term remembers that Priya prefers email in Bengali, so every future ticket starts already knowing it. And remember Session 6's warning — **what you store is a privacy decision**: keep only what the user gave you and only what you need.


## 7. Subgraphs — build big graphs from small ones

As a graph grows, you want the same thing you want in any codebase: **reusable, testable pieces**. A **subgraph** is simply a compiled graph used as a **node** inside a bigger graph. Build "verify a student's identity" once, test it once, and drop it into admissions, grade appeals, and transcript requests.

💡 **Real-life:** a **function you call from many places** — or a reusable "ID check" station that several university offices all send you through.

There are two cases, depending on whether the parent and subgraph share state fields.

Reference: [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs).


### 7.1 Shared state → drop the subgraph in as a node

🎯 **Purpose:** when the subgraph reads and writes the **same fields** as the parent, you add the *compiled subgraph object itself* as a node. State flows straight through — no glue code.

We'll build an **admissions pipeline**: `receive → verify → decide`, where **`verify` is its own graph**.


In [24]:
# --- the reusable subgraph: verify a student's ID ---
class VerifyState(TypedDict):
    student: str
    id_verified: bool
    note: str


def check_id(state: VerifyState) -> dict:
    return {"id_verified": True, "note": f"Verified {state['student']}'s student ID."}


verify_subgraph = (
    StateGraph(VerifyState)
    .add_node("check_id", check_id)
    .add_edge(START, "check_id")
    .add_edge("check_id", END)
    .compile()
)


# --- the parent graph uses it as one node ---
def receive(state: VerifyState) -> dict:
    return {"note": "Application received."}


def decide(state: VerifyState) -> dict:
    return {"note": "Admitted." if state["id_verified"] else "On hold."}


admissions = (
    StateGraph(VerifyState)
    .add_node("receive", receive)
    .add_node("verify", verify_subgraph)   # <-- a whole graph as a single node
    .add_node("decide", decide)
    .add_edge(START, "receive")
    .add_edge("receive", "verify")
    .add_edge("verify", "decide")
    .add_edge("decide", END)
    .compile()
)

print(admissions.invoke({"student": "Aisha"}))

{'student': 'Aisha', 'id_verified': True, 'note': 'Admitted.'}


### 7.2 Peek inside a subgraph while streaming

🎯 **Purpose:** by default streaming hides the subgraph's internal steps. Pass **`subgraphs=True`** to see them too — each event comes with a **namespace** (`ns`) telling you which subgraph it came from (`()` is the parent).


In [25]:
for ns, chunk in admissions.stream({"student": "Ben"}, stream_mode="updates", subgraphs=True):
    where = "parent" if ns == () else f"subgraph {ns[0].split(':')[0]}"
    print(f"[{where}] {chunk}")

[parent] {'receive': {'note': 'Application received.'}}
[subgraph verify] {'check_id': {'id_verified': True, 'note': "Verified Ben's student ID."}}
[parent] {'verify': {'student': 'Ben', 'id_verified': True, 'note': "Verified Ben's student ID."}}
[parent] {'decide': {'note': 'Admitted.'}}


### 7.3 Different state → wrap the subgraph in a node

🎯 **Purpose:** when the subgraph speaks a **different vocabulary** (its own state fields), you can't drop it in directly. Instead, write a node function that **translates**: map parent state → subgraph input, `invoke` it, then map the result back. This also keeps the subgraph's internals private.

Here a credit-lookup subgraph thinks in `code`/`credits`, but the parent thinks in `course`/`summary`.


In [26]:
class InnerState(TypedDict):
    code: str
    credits: int


def compute_credits(state: InnerState) -> dict:
    return {"credits": 3 if state["code"].upper().startswith("CS") else 4}


credit_subgraph = (
    StateGraph(InnerState)
    .add_node("compute_credits", compute_credits)
    .add_edge(START, "compute_credits")
    .add_edge("compute_credits", END)
    .compile()
)


class OuterState(TypedDict):
    course: str
    summary: str


def call_credit_subgraph(state: OuterState) -> dict:
    inner_out = credit_subgraph.invoke({"code": state["course"]})   # translate in
    return {"summary": f"{state['course']} is worth {inner_out['credits']} credits."}  # translate out


outer = (
    StateGraph(OuterState)
    .add_node("credits", call_credit_subgraph)
    .add_edge(START, "credits")
    .add_edge("credits", END)
    .compile()
)

print(outer.invoke({"course": "CS101"}))

{'course': 'CS101', 'summary': 'CS101 is worth 3 credits.'}


| Parent & subgraph share state fields? | How to add it |
|---|---|
| **Yes** (same keys) | `add_node("name", compiled_subgraph)` — pass the subgraph directly |
| **No** (different keys) | Wrap it: a node that maps state in, calls `subgraph.invoke(...)`, maps state out |

**Northstar parallel:** an **identity-verification** subgraph — verify email, check the account is active, confirm recent activity — built and tested once, then reused inside the refund flow, the cancellation flow, and the data-export flow. Fix a bug there and every flow gets the fix.


## 8. Recap

You can now take a graph from "works on my laptop" to "usable by real people":

| Capability | The one line that matters | Use it when… |
|---|---|---|
| **Streaming** | `graph.stream(x, stream_mode="updates" / "messages" / "custom")` | The user shouldn't stare at a blank screen |
| **Event streaming** | `async for e in graph.astream_events(x, version="v2")` | You need tool calls + tokens + steps for a rich UI |
| **Interrupts** | `interrupt(payload)` … `Command(resume=answer)` | A human must approve before the graph continues |
| **Time travel** | `get_state_history` → `invoke(None, cfg)` / `update_state` | Debugging, or "edit-and-rerun" / "what if?" |
| **Memory** | checkpointer + `trim_messages`; store + `runtime.store` | Remember within a chat, and across sessions, without overflowing |
| **Subgraphs** | `add_node("name", compiled_subgraph)` | Reuse a whole graph as a tested building block |

Every one of these is powered by the **checkpointer** from Session 6 — persistence is what makes streaming resumable, interrupts pausable, and time travel possible.

### Check yourself (no notes)

1. Name three `stream_mode` values and what each yields.
2. When would you choose `astream_events` over `stream`?
3. What two things must a graph have before `interrupt()` works — and what re-runs when you resume?
4. What's the difference between **replaying** a checkpoint and **forking** one with `update_state`?
5. Short-term vs long-term memory: which grows and needs trimming, and which crosses threads?
6. You have an ID-check flow that shares state with its parent. How do you add it — directly, or with a wrapper?

### References

- [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming) · [Event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming) · [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel) · [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory) · [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)
